# SSM Occurence Indexer
```
ssm_occurrence{}
            |____ ssm{}
            |        |____ consequence[]
            |                     |_____ transcript{}
            |                                   |_____ gene{}
            |                                   |_____ annotation{}
            |____ case{}
                     |____ observation[]
```

In [22]:
import os
import requests
import uuid
%load_ext autoreload
from exports.builders import MAFBuilder, SSMOccurrenceCentricBuilder, TranscriptBuilder, ObservationBuilder, CaseBuilder
from exports.builders.utils import struct_select, uuid5_col
from exports.mappers import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from config import TestConfig

conf = TestConfig()

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType
%cd /mnt/Projects/gdc-mutation-indexer

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/mnt/Projects/gdc-mutation-indexer


## Load combined maf into spark

In [2]:
 maf_df = MAFBuilder(conf, sqlContext).build()

### SSM

In [3]:
ssm_df = maf_df.select('_case_submitter_id', *struct_select(conf.mappings['ssm']))

### Consequence

In [4]:
cons_df = TranscriptBuilder(conf, sqlContext).build(maf_df, join_gene=True)

### Observation

In [5]:
obs_df = ObservationBuilder(conf, sqlContext).build(maf_df)

### Case

In [6]:
case_df = CaseBuilder(conf, sqlContext).build()

### Case + Obs

In [7]:
case_obs_df = case_df.join(obs_df, case_df.submitter_id == obs_df._case_submitter_id, 'right')\
                .select('submitter_id', 'case_id', 'ssm_id',
                    struct(
                        'observation',
                        *case_df.columns
                    ).alias('case'))\
                .drop('_case_submitter_id')

In [10]:
print 'size of case_obs', case_obs_df.count()
#case_obs_df.printSchema()

size of case_obs 18


### SSM + Consequence

In [18]:
ssm_cons = ssm_df.join(cons_df, on='ssm_id')\
                    .select('ssm_id',
                            struct('consequence',
                                   *ssm_df.drop('_case_submitter_id').columns)\
                            .alias('ssm'))

In [20]:
print 'size of ssm_cons', ssm_cons.count()

size of ssm_cons 18


### Final join

In [25]:
ssm_occurrence_centric = ssm_cons.join(case_obs_df, on='ssm_id', how='right')\
                                    .withColumn('ssm_occurrence_id',
                                                uuid5_col(lit('ssm_occurrence'),
                                                    col('ssm_id'),
                                                    col('case_id')))\
                                    .drop('ssm_id')\
                                    .drop('case_id')\
                                    .drop('submitter_id')

In [30]:
ssm_occurrence_centric.printSchema()

root
 |-- ssm: struct (nullable = true)
 |    |-- consequence: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- transcript: struct (nullable = false)
 |    |    |    |    |-- annotation: struct (nullable = true)
 |    |    |    |    |    |-- impact: string (nullable = true)
 |    |    |    |    |    |-- amino_acids: string (nullable = true)
 |    |    |    |    |    |-- existing_variation: string (nullable = true)
 |    |    |    |    |    |-- sift: string (nullable = true)
 |    |    |    |    |    |-- pubmed: string (nullable = true)
 |    |    |    |    |    |-- dbsnp_val_status: string (nullable = true)
 |    |    |    |    |    |-- ccds: string (nullable = true)
 |    |    |    |    |    |-- cdna_position: string (nullable = true)
 |    |    |    |    |    |-- cds_start: string (nullable = true)
 |    |    |    |    |    |-- hgvsp: string (nullable = true)
 |    |    |    |    |    |-- ensp: string (nullable = true)
 |    |    |    |

In [29]:
print 'size of ssm_occurrence', ssm_occurrence_centric.count()

size of ssm_occurrenc 18
